In [ ]:
import os
import math
import random
import pickle
import zipfile
import textwrap
import numpy as np
import seaborn as sns
import plotly.offline as py
import plotly.colors as plc
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from scipy.interpolate import CubicSpline

SEED = 96
PRECISION = 4
np.set_printoptions(precision=PRECISION, suppress=True, floatmode="fixed")

## Shared Utilities / Helpers

#### Linear algebra helpers:

In [ ]:
def frobenius_norm_sq(A):
    """
    Frobenius norm squared of a matrix.
    """
    A = np.asarray(A, dtype=float)
    return float(np.sum(np.square(A)))

In [ ]:
def l2_norm(v):
    """
    L2 norm of a vector.
    """
    v = np.asarray(v, dtype=float)
    return float(np.linalg.norm(v))

In [ ]:
def orthogonalize(v, basis_vectors):
    """
    Gram-Schmidt orthogonalization + normalization of a vector with respect to a list of basis vectors.

    Parameters:
    v (list of floats): Vector to orthogonalize.
    basis_vectors (list of lists of floats): List of basis vectors.

    Returns:
    numpy.ndarray | None: Orthogonalized unit vector, or None if degenerate.
    """
    v_new = np.asarray(v, dtype=float).copy()

    for b in basis_vectors:
        b_np = np.asarray(b, dtype=float)
        norm_sq = float(np.dot(b_np, b_np))
        if norm_sq < 1e-12:
            continue

        proj = float(np.dot(v_new, b_np)) / norm_sq
        v_new -= proj * b_np

    norm = float(l2_norm(v_new))
    if norm < 1e-12:
        return None

    return v_new / norm

In [ ]:
def power_iteration(matrix, max_iterations=100, tolerance=1e-7, basis_vectors=None):
    """
    Computes largest eigenvalue and corresponding eigenvector using power iteration.

    Parameters:
    matrix (list of lists of floats): Input square matrix.
    max_iterations (int): Number of allowed iterations.
    tolerance (float): Convergence tolerance.
    basis_vectors (list): Optional basis vectors for orthogonalized starts.

    Returns:
    top_eigval (float): Largest eigenvalue.
    top_eigvec (numpy.ndarray): Corresponding eigenvector.
    """
    A = np.asarray(matrix, dtype=float)
    n = A.shape[1]

    if basis_vectors is None:
        basis_vectors = []

    # initialize random vector and orthogonalize (this will be prev eigenvector)
    v_prev = np.random.standard_normal(n)
    v_prev = orthogonalize(v_prev, basis_vectors)

    if v_prev is None:
        fallback = np.random.standard_normal(n)
        fallback /= max(float(l2_norm(fallback)), 1e-12)
        return 0.0, fallback

    for _ in range(max_iterations):
        # matrix-vector multiplication (this is the current eigenvector)
        v_curr = A @ v_prev

        # keep iterating orthogonal to known basis when provided
        if basis_vectors:
            v_curr = orthogonalize(v_curr, basis_vectors)
            if v_curr is None:
                return 0.0, v_prev

        # normalize curr eigenvector and handle zero vec case
        norm = float(l2_norm(v_curr))
        if norm < 1e-12:
            return 0.0, v_prev
        v_curr = v_curr / norm

        # sign-invariant convergence check
        delta = min(
            float(l2_norm(v_curr - v_prev)),
            float(l2_norm(v_curr + v_prev)),
        )

        # update previous eigenvector
        v_prev = v_curr

        # check convergence (delta should approach 0)
        if delta < tolerance:
            break

    # compute corresponding eigenvalue (Rayleigh quotient)
    top_eigval = float(v_prev @ (A @ v_prev))
    top_eigvec = v_prev

    return top_eigval, top_eigvec

In [ ]:
def top_k_eigenpairs(
    matrix,
    max_k=30,
    max_subspace_iterations=500,
    tolerance=1e-9,
):
    """
    Top k eigenpairs of symmetric A via simultaneous (subspace) iteration + Rayleigh–Ritz.

    Orthonormal eigenvectors; avoids deflation drift when k is large.

    Parameters:
    matrix: Square symmetric matrix (list of lists or ndarray).
    max_k: Number of eigenpairs to return (capped at matrix size).
    max_subspace_iterations: QR iterations on span(A^k Q).
    tolerance: Stop when max off-diagonal of the Rayleigh matrix H = QᵀAQ is below this.

    Returns:
    eigvals (np.ndarray): Top eigenvalues in descending order, shape (p,).
    eigvecs (np.ndarray): Corresponding eigenvectors as columns, shape (n, p).
    """
    A = np.asarray(matrix, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("matrix must be square")

    n = A.shape[0]
    p = min(max_k, n)
    if p == 0:
        return np.empty((0,), dtype=float), np.empty((n, 0), dtype=float)

    # initialize random orthonormal subspace basis
    Q, _ = np.linalg.qr(np.random.standard_normal((n, p)), mode="reduced")

    # iterate QR iterations on span(A^k Q)
    # until reduced problem is nearly diagonal
    for _ in range(max_subspace_iterations):

        Z = A @ Q
        Q_new, _ = np.linalg.qr(Z, mode="reduced")

        # Rayleigh–Ritz projection
        H = Q_new.T @ A @ Q_new
        H = 0.5 * (H + H.T)

        # check convergence
        off = float(np.max(np.abs(H - np.diag(np.diag(H)))))
        Q = Q_new
        if off < tolerance:
            break

    # final Rayleigh–Ritz projection
    H = Q.T @ A @ Q
    H = 0.5 * (H + H.T)

    # compute eigenvalues and eigenvectors
    eigvals_h, eigvecs_h = np.linalg.eigh(H)

    # sort eigenvalues in descending order
    idx = np.argsort(eigvals_h)[::-1]
    eigvals = eigvals_h[idx][:p]

    # reduce eigenvectors to top-k
    vecs_reduced = eigvecs_h[:, idx][:, :p]
    eigvecs = Q @ vecs_reduced

    # normalize eigenvectors
    col_norms = np.linalg.norm(eigvecs, axis=0, keepdims=True)
    col_norms[col_norms < 1e-12] = 1.0
    eigvecs = eigvecs / col_norms

    return eigvals, eigvecs

In [ ]:
def sing_val_decomp(matrix, r=None, mode="full"):
    """
    Manually performs singular value decomposition (SVD) on a given matrix.
    
    Parameters:
    matrix (list of lists of floats): Input matrix to decompose.
    r (int): Number of singular values to compute.
    mode (str): "full" or "partial".

    Returns:
    U   (list of lists of floats): Left singular vectors.
    S   (list of lists of floats): Singular values.
    V_T (list of lists of floats): Right singular vectors (transposed).
    """
    M = np.asarray(matrix, dtype=float)
    m, n = M.shape
    r = min(m, n) if r is None else min(int(r), m, n)

    # compute right-side positive semidefinite matrix
    PSD_R = M.T @ M

    eigenvalues, eigenvectors = top_k_eigenpairs(PSD_R, max_k=r)

    # compute U (m, r), V (n, r), and S (r, r)
    if mode == "partial":
        U, V_T = None, None
        S = np.zeros((r, r), dtype=float)
    elif mode == "full":
        U = np.zeros((m, r), dtype=float)
        V = np.zeros((n, r), dtype=float)
        S = np.zeros((r, r), dtype=float)
    else:
        raise ValueError("Invalid mode. Must be 'full' or 'partial'.")

    eps = 1e-10
    for i in range(r):
        # compute sigma_i
        sigma = max(float(eigenvalues[i]), 0.0) ** 0.5
        S[i, i] = sigma if sigma > eps else 0.0

        if mode == "full":
            # compute V_i
            eigvec_i = np.asarray(eigenvectors[:, i], dtype=float)
            V[:, i] = eigvec_i

            # compute U_i = (M * V_i) / sigma_i
            if sigma > eps:
                U[:, i] = (M @ eigvec_i) / sigma

    if mode == "full":
        V_T = V.T

    return U, S, V_T

#### Evaluation helpers:

In [ ]:
def get_padded_range(vmin, vmax, tick_step):
    """Helper to add  padding so lines don't hit absolute edge of plot box."""
    return [vmin - tick_step, vmax + tick_step]

In [ ]:
def wrap_text(text, width=16):
    """Wraps text to a given width."""
    return ["\n".join(textwrap.wrap(s, width=width)) for s in text]

In [ ]:
def interpolate_sequence(data_sequence, num_interpolations=10):
    """
    Interpolates a sequence of matrices along a time axis using cubic splines.
    
    Parameters:
    data_sequence (list of numpy.ndarray): List of matrices representing data at different time steps.
    num_interpolations (int): Number of interpolation steps between each pair of matrices.
    
    Returns:
    interpolated_data (numpy.ndarray): Interpolated data sequence of shape (total_steps, N, D).
    """
    # convert list of decade matrices to 3D array
    data = np.asarray(data_sequence, dtype=float)
    T, N, D = data.shape

    # parameterize time steps t from 0 to T-1
    t = np.arange(T)

    # compute total number of interpolation steps
    # and create dense time grid for interpolation
    total_steps = (T - 1) * (num_interpolations + 1) + 1
    t_interp = np.linspace(0, T-1, total_steps)

    # initialize cubic spline interpolation on time axis
    # and set natural boundary conditions (so curves are less likely to overshoot)
    spline = CubicSpline(t, data, axis=0, bc_type="natural")
    return spline(t_interp)

In [ ]:
def cosine_similarity(v1, v2):
    """
    Computes the cosine similarity between two vectors.
    
    Parameters:
    v1, v2: Input vectors.

    Returns:
    float: Cosine similarity between the two vectors.
    """
    v1 = np.asarray(v1, dtype=float)
    v2 = np.asarray(v2, dtype=float)

    # calculate the dot product
    dot = float(np.dot(v1, v2))
    
    # calculate L2 norm for each vector
    norm_v1 = float(np.linalg.norm(v1))
    norm_v2 = float(np.linalg.norm(v2))

    # handle division by zero if any of the vectors have zero magnitude
    if norm_v1 == 0 or norm_v2 == 0:
        return 0

    # calculate cosine similarity
    return dot / (norm_v1 * norm_v2)

In [ ]:
def jaccard_similarity(set1, set2):
    """
    Calculate the Jaccard similarity between two sets.
    
    Parameters:
    set1, set2: Input sets to compare.

    Returns:
    float: Jaccard similarity between the two sets.
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

## Step 1: Preprocessing

#### Load HistWords dataset (uncomment on first run):

In [ ]:
# # dataset obtained from here: https://github.com/williamleif/histwords

# # load dataset here:
# !git clone https://github.com/williamleif/histwords.git
# !wget -O all_english_embeddings.zip "http://snap.stanford.edu/historical_embeddings/eng-all_sgns.zip"

# # unzip loaded dataset
# with zipfile.ZipFile("all_english_embeddings.zip", 'r') as zip_ref:
#     zip_ref.extractall("all_english_embeddings")

In [ ]:
# base dir for embeddings
base_dir = "all_english_embeddings/sgns"
print(os.getcwd()) # debugging

In [ ]:
# for loading historical embeddings from .npy files
def load_embeddings(file_path):
    return np.load(file_path)

#### Obtain sample of words from dataset:

In [ ]:
# specify decades of interest
decades = ['1880', '1890', '1900', '1910', '1920', '1930', '1940', '1950', '1960', '1970', '1980']
vocab_dict = {}
embeddings_dict = {}
random.seed(SEED)       # set desired seed here
sample_size = 500       # set desired sample size here

In [ ]:
# load and optionally sample each decade's embeddings and vocabulary
for decade in decades:
    # load embeddings
    embedding_path = os.path.join(base_dir, f"{decade}-w.npy")
    embeddings = load_embeddings(embedding_path)
    
    # load vocabulary
    vocab_path = os.path.join(base_dir, f"{decade}-vocab.pkl")
    with open(vocab_path, 'rb') as f:
        vocab = pickle.load(f)
    
    # ensure embeddings and vocab are same size
    assert len(embeddings) == len(vocab), f"Mismatch in size for {decade}"

    # filter out any numerical values incorrectly counted as "words"
    filtered_vocab = []
    filtered_embeddings = []
    for word, embedding in zip(vocab, embeddings):
        if not any(char.isdigit() for char in word):
            filtered_vocab.append(word)
            filtered_embeddings.append(embedding)

    # fill dicts
    vocab_dict[decade] = filtered_vocab
    embeddings_dict[decade] = np.array(filtered_embeddings)

#### Filter subsamples by intersection and non-zero vectors across decades:

In [ ]:
# obtain common vocabulary across all decades
candidate_vocabs = set(vocab_dict[decades[0]])
for decade in decades[1:]:
    candidate_vocabs.intersection_update(vocab_dict[decade])

# convert to list for indexing, sort for determinism, then shuffle for reproducible randomness
candidate_vocabs = list(candidate_vocabs)
candidate_vocabs.sort()
random.shuffle(candidate_vocabs)

In [ ]:
# precompute word to index dictionaries for each decade
word_to_idx_dict = {}
for decade in decades:
    vocab = vocab_dict[decade]
    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    word_to_idx_dict[decade] = word_to_idx

valid_words = []
zero_vector = np.zeros(embeddings_dict[decades[0]].shape[1])

# filter out words with zero vectors across all decades
for word in candidate_vocabs:
    has_zero_vector = False
    for decade in decades:
        idx = word_to_idx_dict[decade][word]
        embedding = embeddings_dict[decade][idx]
        if np.array_equal(embedding, zero_vector):
            has_zero_vector = True
            break

    # add word to valid_words if it has non-zero vectors across all decades
    if not has_zero_vector:
        valid_words.append(word)

    # stop if we have enough valid words
    if len(valid_words) == sample_size:
        break

print(f"Number of valid words (non-zero embeddings in all decades): {len(valid_words)}")

# update vocab_dict and embeddings_dict
for decade in decades:
    embeddings = embeddings_dict[decade]
    word_to_idx = word_to_idx_dict[decade]
    
    # obtain embeddings for valid_words
    indices = [word_to_idx[word] for word in valid_words]
    filtered_embeddings = embeddings[indices]
    
    # update dicts with filtered values
    vocab_dict[decade] = valid_words
    embeddings_dict[decade] = filtered_embeddings

# rebuild word to index dict after subsampling
word_to_idx_dict = {
    decade: {w: i for i, w in enumerate(vocab_dict[decade])}
    for decade in decades
}

In [ ]:
# debugging
for decade, embeddings in embeddings_dict.items():
    print(f"{decade}'s embeddings:")
    print(f"shape: {embeddings.shape}") # display shape of embeddings per decade
    print(embeddings_dict[decade][:5])  # display first n rows of embeddings per decade
    print()

## Step 2: Standardization

In [ ]:
def standardize(data):
    """
    Standardizes vector data by centering (subtracting mean) and scaling (dividing by standard deviation).

    Parameters:
    data (numpy.ndarray): Input data matrix of shape (num_vectors, num_dimensions).

    Returns:
    standardized_data (numpy.ndarray): Standardized data by shifting to zero mean and scaling to unit variance.
    mean_array (numpy.ndarray): Computed mean matrix.
    std_array (numpy.ndarray): Computed standard deviation matrix.
    """
    data = np.asarray(data, dtype=float)

    # compute mean and std for each dimension
    # using ddof=1 for sample std, Bessel's correction (unbiased estimator)
    mean_array = np.mean(data, axis=0)
    std_array = np.std(data, axis=0, ddof=1)

    # center and scale data
    standardized_data = (data - mean_array) / np.where(std_array != 0, std_array, 1.0)
    return standardized_data, mean_array, std_array

#### Center each decade's high-dim point cloud to zero mean and scale to unit variance:

In [ ]:
standardized_embeddings_dict = {}
for decade, embeddings in embeddings_dict.items():
    standardized_embeddings = standardize(embeddings_dict[decade])[0]
    standardized_embeddings_dict[decade] = standardized_embeddings

In [ ]:
# debugging: per decade — non-centered vs centered mean (first 5 dimensions)
for decade in decades:
    mean_before = standardize(embeddings_dict[decade])[1]
    mean_after = standardize(standardized_embeddings_dict[decade])[1]
    std_before = standardize(embeddings_dict[decade])[2]
    std_after = standardize(standardized_embeddings_dict[decade])[2]
    print(f"{decade}'s embedding means / stds before vs. after standardization (first 5 dims):")
    print(f"  Before: {mean_before[:5]} / {std_before[:5]}")
    print(f"  After : {mean_after[:5]} / {std_after[:5]}")
    print()

## Step 3: Alignment

In [ ]:
def procrustes_residual(A, B, norm_A, norm_B):
    """
    Manually computes the Procrustes residual between two matrices using singular values.
    
    Parameters:
    A (np.ndarray): First matrix, shape (n, d).
    B (np.ndarray): Second matrix, shape (n, d).
    norm_A (float): Frobenius norm squared of A.
    norm_B (float): Frobenius norm squared of B.
    
    Returns:
    float: Procrustes residual between A and B.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    M = B.T @ A

    # sum of singular values of cross-covariance (same as trace term in optimal orthogonal alignment)
    trace_sigma = float(np.sum(np.linalg.svd(M, full_matrices=False, compute_uv=False)))

    # compute residual
    residual = norm_A + norm_B - 2.0 * trace_sigma
    return max(0.0, float(residual))

In [ ]:
def build_pairwise_residual_matrix(embeddings_dict, decades, norm_cache):
    """
    Builds a pairwise residual matrix for a given set of decades.

    Parameters:
    embeddings_dict (dict): Dictionary of embeddings for each decade.
    decades (list of str): List of decades to compute pairwise residuals for.
    norm_cache (dict): Dictionary of Frobenius norms for each decade.

    Returns:
    R (numpy.ndarray): Pairwise residual matrix.
    """
    m = len(decades)
    R = np.zeros((m, m), dtype=float)

    # cache per-decade matrices/transposes once to reduce repeated conversion and transpose overhead
    mats = {d: np.asarray(embeddings_dict[d], dtype=float) for d in decades}
    mats_T = {d: mats[d].T for d in decades}

    for i in range(m):
        d1 = decades[i]
        A = mats[d1]
        for j in range(i + 1, m):
            d2 = decades[j]
            B = mats[d2]
            B_T = mats_T[d2]

            residual = procrustes_residual(A, B, norm_cache[d1], norm_cache[d2])

            R[i, j] = residual
            R[j, i] = residual
    return R

In [ ]:
def compute_transform_matrix(A, B):
    """
    Orthogonal Procrustes rotation W minimizing:
    ||A W - B||_F (same width).

    After SVD, factors M = AᵀB as U Σ Vᵀ. The resulting orthogonal alignment map is W = U Vᵀ
    (minimizes ‖A W - B‖_F for an orthogonal W).

    Parameters:
    A (np.ndarray): Matrix to transform.
    B (np.ndarray): Target matrix.

    Returns:
    W (np.ndarray): Orthogonal Procrustes rotation matrix.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    M = A.T @ B

    U, _, V_T = np.linalg.svd(M, full_matrices=False, compute_uv=True)
    W = U @ V_T
    return W

In [ ]:
# compute Frobenius norms for each decade
norm_cache = {
    decade: frobenius_norm_sq(embeddings)
    for decade, embeddings in standardized_embeddings_dict.items()
}

In [ ]:
# compute pairwise Procrustes residuals
R = build_pairwise_residual_matrix(standardized_embeddings_dict, decades, norm_cache)

# compute anchor scores
anchor_scores = {} # decade to score dict
for i, decade in enumerate(decades):
    anchor_scores[decade] = sum(R[i])

# pick lowest scoring residual decade as the anchor
anchor_decade = min(anchor_scores, key=anchor_scores.get)

In [ ]:
# compute alignment transforms
transforms = {}
B = standardized_embeddings_dict[anchor_decade]
for decade, embeddings in standardized_embeddings_dict.items():
    if decade == anchor_decade:
        transforms[decade] = None
        continue
    A = embeddings
    W = compute_transform_matrix(A, B)
    transforms[decade] = W

In [ ]:
# apply alignment transforms to align all embeddings to anchor decade
aligned_embeddings_dict = {}
for decade, embeddings in standardized_embeddings_dict.items():
    A = embeddings
    W = transforms[decade]
    if W is None:
        aligned_embeddings_dict[decade] = A.copy() # deep copy of matrix to avoid modifying original
    else:
        aligned_embeddings_dict[decade] = A @ W

In [ ]:
anchor_matrix = np.asarray(standardized_embeddings_dict[anchor_decade], dtype=float)
print(f"Anchor decade: {anchor_decade}\n")

# debugging: per decade — alignment error vs anchor before and after Procrustes
pre_alignment_errors = []
post_alignment_errors = []
for decade in decades:
    if decade != anchor_decade:
        A_pre = np.asarray(standardized_embeddings_dict[decade], dtype=float)
        A_post = np.asarray(aligned_embeddings_dict[decade], dtype=float)
        pre_err = frobenius_norm_sq(A_pre - anchor_matrix)
        post_err = frobenius_norm_sq(A_post - anchor_matrix)
        pre_alignment_errors.append(pre_err)
        post_alignment_errors.append(post_err)
        print(f"{decade} vs {anchor_decade} alignment errors:")
        print(f"  Before: {pre_err:.{PRECISION}f}")
        print(f"  After:  {post_err:.{PRECISION}f}")
        print()

improved_count = sum(
    1 for pre, post in zip(pre_alignment_errors, post_alignment_errors) if post < pre
)
avg_pre = sum(pre_alignment_errors) / len(pre_alignment_errors)
avg_post = sum(post_alignment_errors) / len(post_alignment_errors)

print(f"Decades improved (lower error after): {improved_count}/{len(decades) - 1}")
print(f"Average pre-alignment squared error:  {avg_pre:.{PRECISION}f}")
print(f"Average post-alignment squared error: {avg_post:.{PRECISION}f}")

## Step 4: Reduction

#### Calculate covariance matrix and find top-k eigenvalues:

In [ ]:
def compute_cv_matrix(data):
    """
    Manually computes covariance matrix of input data.
    
    Parameters:
    data (list of lists of floats): Centered data matrix of shape (num_vectors, num_dimensions).
    
    Returns:
    covariance_matrix (list of lists of floats): Covariance matrix of shape (num_dimensions, num_dimensions).
    """
    A = np.asarray(data, dtype=float)
    num_vectors = A.shape[0]
    covariance_matrix = (A.T @ A) / (num_vectors - 1)
    return covariance_matrix

In [ ]:
# combine embeddings from all decades into one dataset for unified coordinate space
combined_embeddings = np.vstack([aligned_embeddings_dict[decade] for decade in decades])

# sanity check: mean of each decade's embeddings should be close to zero
mu = combined_embeddings.mean(axis=0)
is_centered = np.allclose(mu, 0.0, atol=1e-8) # if needed, we can relax atol here
assert is_centered, "Combined embeddings are not centered"
print(f"Is centered? {is_centered}")

In [ ]:
# find cv matrix of combined embeddings
cv_matrix = compute_cv_matrix(combined_embeddings)

# sanity check: the cv matrix should be symmetric
is_symmetric = np.allclose(cv_matrix, cv_matrix.T)
assert is_symmetric, "CV matrix is not symmetric"
print(f"Is symmetric? {is_symmetric}")

In [ ]:
# debugging
largest_eigenvalue, eigenvector = power_iteration(cv_matrix)
print("Largest eigenvalue:", largest_eigenvalue)

# debugging
k = 3
eigenvalues, eigenvectors = top_k_eigenpairs(cv_matrix, k)
print(f"Largest {k} eigenvalues:", eigenvalues)

#### Perform PCA projection:

In [ ]:
# project each decade's aligned embeddings onto the common pooled PC basis (columns of eigenvectors)
proj_embeddings_dict = {} # this dict should hold embeddings of words in R3
for decade in decades:
    projected_embeddings = aligned_embeddings_dict[decade] @ eigenvectors
    proj_embeddings_dict[decade] = projected_embeddings

In [ ]:
# check for matching vocab and projected embedding dict lengths (debugging)
for decade in decades:
    assert len(vocab_dict[decade]) == len(embeddings_dict[decade]), f"Mismatch in vocab and embeddings for {decade}"

In [ ]:
print("Keys in projected embeddings dict:", proj_embeddings_dict.keys())

## Step 5: Evaluation

#### Quantify trajectorial behavior of select words using Euclidean and Cosine distances:

In [ ]:
# assemble R300 and R3 word trajectories across aligned embedding spaces

word_trajectories = {}
proj_word_trajectories = {}

for word in valid_words:
    trajectory = []
    proj_trajectory = []

    for decade in decades:
        idx = word_to_idx_dict[decade][word]

        r300_vec = aligned_embeddings_dict[decade][idx]
        r3_vec = proj_embeddings_dict[decade][idx]

        trajectory.append(r300_vec)
        proj_trajectory.append(r3_vec)

    word_trajectories[word] = np.array(trajectory)
    proj_word_trajectories[word] = np.array(proj_trajectory)

print(f"Assembled trajectories for {len(word_trajectories)} words.")

In [ ]:
# calculate three word-independent metrics on R300 space word trajectories

# word to score pairs
path_instability_scores = {}   # path instability score for each word (std dev of step magnitudes)
cumulative_dist_scores = {}    # cumulative straight-line path length score for each word (Euclidean distance)
displacement_scores = {}       # end-to-end angular displacement score for each word (cosine distance)

# calculate shifts for all words across decades
for word, trajectory in word_trajectories.items():
    # find stepwise differences/magnitudes between consecutive decades
    step_deltas = np.diff(trajectory, axis=0)
    step_magnitudes = np.linalg.norm(step_deltas, axis=1)

    # calculate end-to-end angular displacement
    start_vec = trajectory[0]
    end_vec = trajectory[-1]
    displacement = 1 - cosine_similarity(start_vec, end_vec)

    path_instability_scores[word] = float(np.std(step_magnitudes, ddof=1))
    cumulative_dist_scores[word] = float(np.sum(step_magnitudes))
    displacement_scores[word] = float(displacement)

# sort words by volatility in descending order
sorted_path_instability = sorted(path_instability_scores.items(), key=lambda x: x[1], reverse=True)
sorted_cumulative_dist = sorted(cumulative_dist_scores.items(), key=lambda x: x[1], reverse=True)
sorted_displacement = sorted(displacement_scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# plot trajectory metrics for top and bottom N words

N = 15

trajectory_metrics_data = [
    ("Path Instability (Euclidean)", sorted_path_instability, path_instability_scores, 'salmon',    'cadetblue'),
    ("Cumulative Path (Euclidean)",  sorted_cumulative_dist,  cumulative_dist_scores,  'salmon',    'cadetblue'),
    ("Net Displacement (Cosine)",    sorted_displacement,     displacement_scores,     'salmon',    'cadetblue'),
]

fig, ax = plt.subplots(2, 3, figsize=(15, 8))

for col, (label, sorted_scores, score_dict, top_color, bot_color) in enumerate(trajectory_metrics_data):
    top_words_m, top_scores_m = zip(*sorted_scores[:N])
    bot_words_m, bot_scores_m = zip(*sorted_scores[-N:])

    x_max = max(top_scores_m) * 1.08
    x_min = 0

    # top row
    ax[0, col].barh(top_words_m, top_scores_m, color=top_color)
    ax[0, col].invert_yaxis()
    ax[0, col].set_title(f"Top {N}")
    ax[0, col].set_xlabel("Score")
    ax[0, col].set_xlim(x_min, x_max)
    ax[0, col].axvline(0, color='black', linewidth=1)

    # bottom row
    ax[1, col].barh(bot_words_m, bot_scores_m, color=bot_color)
    ax[1, col].invert_yaxis()
    ax[1, col].set_title(f"Bottom {N}")
    ax[1, col].set_xlabel("Score")
    ax[1, col].set_xlim(x_min, x_max)
    ax[1, col].axvline(0, color='black', linewidth=1)

fig.subplots_adjust(top=0.88, hspace=0.35, wspace=0.6)
fig.suptitle(f"Trajectory Metrics for Top & Bottom {N}-Word Subset", y=1.01, fontsize=16)

# place metric labels using final axis positions
for col, (label, *_) in enumerate(trajectory_metrics_data):
    pos = ax[0, col].get_position() # figure coords, after subplots_adjust
    fig.text(
        (pos.x0 + pos.x1) / 2,
        pos.y1 + 0.055, # above "Top N" title
        label,
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
        transform=fig.transFigure,
    )

plt.show()

#### Quantify changes in neighborhood overlap of select words using k-NN and Jaccard similarity:

In [ ]:
K = 10 # number of nearest neighbors to consider
neighbor_sets = {d: {} for d in decades} # word to per-decade neighbor set pairs
neighbor_distances = {d: {} for d in decades} # word to per-decade neighbor distance pairs

# compute neighborhood sets and cosine distances for each word in each decade
# working in Procrustes-aligned, per-decade standardized R300 space

for decade in decades:
    # get aligned embeddings and vocabs for current decade
    X = aligned_embeddings_dict[decade]
    words = vocab_dict[decade]

    # normalize embeddings to unit length
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    Xn = X / norms

    # compute pairwise cosine similarity matrix
    S = Xn @ Xn.T

    # find indices of K nearest neighbors for each word
    knn_idx = np.argsort(S, axis=1)[:, -(K+1):-1]

    # compute neighbor sets and distances for each word
    for i, word in enumerate(words):
        # get indices of K nearest neighbors for current word
        neighbors = {words[j] for j in knn_idx[i]}

        # compute mean cosine distance of current word to its neighbors
        # (distance = 1 - cos similarity)
        avg_dist = 1.0 - np.mean(S[i, knn_idx[i]])

        # store neighbor sets and cosine distances for current word
        neighbor_sets[decade][word] = neighbors
        neighbor_distances[decade][word] = avg_dist

In [ ]:
# calculate two word-neighborhood metrics on R300 space aligned word embeddings

# word to score pairs
turnover_series = {}          # per-decade-pair list (for heatmaps)
dispersion_series = {}        # per-decade-pair list (for heatmaps)
turnover_scores = {}          # full timespan global scalar per word
dispersion_scores = {}        # full timespan global scalar per word

# calculate neighborhood metrics for all words across decades
for word in valid_words:
    turnover_series[word] = []
    dispersion_series[word] = []

    for i in range(len(decades) - 1):
        # get neighbor sets/distances for current word
        dec1, dec2 = decades[i], decades[i + 1]
        set1 = neighbor_sets[dec1].get(word, set())
        set2 = neighbor_sets[dec2].get(word, set())
        dist1 = neighbor_distances[dec1].get(word, 0)
        dist2 = neighbor_distances[dec2].get(word, 0)

        # calculate neighborhood overlap and dispersion scores
        overlap = jaccard_similarity(set1, set2)
        turnover_series[word].append(1.0 - overlap)
        dispersion_series[word].append((dist1 + dist2) / 2.0)

    turnover_scores[word] = float(np.mean(turnover_series[word]))
    dispersion_scores[word] = float(np.mean(dispersion_series[word]))

# sort words by neighborhood volatility in descending order
sorted_turnover = sorted(turnover_scores.items(), key=lambda x: x[1], reverse=True)
sorted_dispersion = sorted(dispersion_scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# plot neighborhood metrics for top and bottom N words

N = 15

decades_labels = [f"{decades[i]}-{decades[i + 1]}" for i in range(len(decades) - 1)]

neighborhood_metrics_data = [
    ("Neighborhood Turnover (1 - Jaccard)", sorted_turnover,    turnover_series,    "YlGnBu"),
    ("Neighborhood Dispersion (Cosine)",    sorted_dispersion,  dispersion_series,  "YlOrRd"),
]

fig, ax = plt.subplots(2, 2, figsize=(16, 11))

for col, (label, sorted_scores, series_dict, cmap) in enumerate(neighborhood_metrics_data):
    top_words_m, top_scores_m = zip(*sorted_scores[:N])
    bot_words_m, bot_scores_m = zip(*sorted_scores[-N:])

    panels = [
        (0, top_words_m, f"Top {N}"),
        (1, bot_words_m, f"Bottom {N}"),
    ]

    for row, words, row_title in panels:
        data = np.asarray([series_dict[w] for w in words], dtype=float)

        sns.heatmap(
            data,
            ax=ax[row, col],
            cmap=cmap,
            annot=True,
            fmt=".2f",
            annot_kws={"size": 7},
            cbar_kws={"shrink": 0.85},
            xticklabels=decades_labels,
            yticklabels=words,
        )
        ax[row, col].set_title(row_title, fontsize=11, pad=8)
        ax[row, col].tick_params(axis="x", labelsize=7, rotation=45)
        ax[row, col].tick_params(axis="y", labelsize=8)

fig.subplots_adjust(top=0.92, hspace=0.25, wspace=0.2)
fig.suptitle(f"Neighborhood Metrics Over Time for Top & Bottom {N}-Word Subset", y=1.01, fontsize=16)

for col, (label, *_) in enumerate(neighborhood_metrics_data):
    pos = ax[0, col].get_position()
    fig.text(
        (pos.x0 + pos.x1) / 2,
        pos.y1 + 0.04,
        label,
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        transform=fig.transFigure,
    )

plt.show()

#### Consolidate metrics for a ranked composite semantic dynamics index using PCA

In [ ]:
# assemble our five feature metrics into single composite correlation matrix

score_features = []
metrics = [
    "Path Instability (Euclidean)", 
    "Cumulative Path (Euclidean)", 
    "Net Displacement (Cosine)", 
    "Neighborhood Turnover (1-Jaccard)",
    "Neighborhood Dispersion (Cosine)"
]

for word in valid_words:
    row = [
        path_instability_scores[word],
        cumulative_dist_scores[word],
        displacement_scores[word],
        turnover_scores[word],
        dispersion_scores[word],
    ]
    score_features.append(row)

feature_matrix = np.array(score_features)
corr_matrix = np.corrcoef(feature_matrix, rowvar=False)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    xticklabels=wrap_text(metrics),
    yticklabels=wrap_text(metrics),
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Metric Correlations", fontsize=12, pad=16)
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# standardize feature matrix and use SVD with PCA to extract dominant semantic variation

r = len(metrics)

# standardize feature matrix
feature_matrix_std = standardize(feature_matrix)[0]

# perform SVD to get 4 singular values for explained variance
# U, S, V_T = np.linalg.svd(feature_matrix_std, full_matrices=True)
U, S, V_T = sing_val_decomp(feature_matrix_std, r=r, mode="full")

# calculate explained variance for first principal component 
# sigma_sq = S**2
sigma_sq = [S[i][i]**2 for i in range(r)]
total_var = sum(sigma_sq)
explained_pc1_var = sigma_sq[0] / total_var

print(f"Variance explained by PC1: {explained_pc1_var * 100:.2f}%\n")

In [ ]:
# extract first principal component vector and plot its loadings

# principal component direction is arbitrary,
# flipping here for consistency
pc1 = V_T[0, :]
if np.sum(pc1) < 0: pc1 = -pc1

plt.figure(figsize=(8, 5))
plt.bar(wrap_text(metrics), pc1, color='salmon')
plt.axhline(0, color='black', linewidth=1)
plt.title("Metric Loadings on PC1", fontsize=12)
plt.ylabel("Weight")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# plot composite index scores for all words (unranked + ranked)

composite_scores = feature_matrix_std @ pc1
words = np.asarray(valid_words, dtype=object)
scores = np.asarray(composite_scores, dtype=float)

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-0.08, 0.08, size=len(scores))

sort_idx = np.argsort(scores)[::-1]
scores_sorted = scores[sort_idx]
words_sorted = words[sort_idx]
ranks = np.arange(1, len(scores) + 1)

interval_step = 0.5
half_extent = float(np.max(np.abs(scores)))
max_score_rounded = math.ceil(half_extent / interval_step) * interval_step
y_pad = max(interval_step, 0.08 * max_score_rounded)
y_min = -max_score_rounded - y_pad
y_max = max_score_rounded + y_pad

marker_size = 3
point_color = "blue"

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    shared_yaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Unranked", "Ranked"),
)

# top: score on y, jitter on x
fig.add_trace(
    go.Scatter(
        x=jitter,
        y=scores,
        mode="markers",
        marker=dict(size=marker_size, color=point_color),
        text=words,
        hovertemplate="Word: %{text}<br>Score: %{y:.4f}<extra></extra>",
        name="Unranked",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# bottom: rank on x, score on y
fig.add_trace(
    go.Scatter(
        x=ranks,
        y=scores_sorted,
        mode="markers",
        marker=dict(size=marker_size, color=point_color),
        text=words_sorted,
        hovertemplate="Rank: %{x}<br>Word: %{text}<br>Score: %{y:.4f}<extra></extra>",
        name="Ranked",
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="",
    showticklabels=False,
    zeroline=False,
    row=1,
    col=1,
)
fig.update_xaxes(title_text="Rank", row=2, col=1)

fig.update_yaxes(title_text="Composite Score (PC1)", range=[y_min, y_max], row=1, col=1)
fig.update_yaxes(title_text="Composite Score (PC1)", range=[y_min, y_max], row=2, col=1)

fig.add_hline(y=0, line_width=1, line_color="black", row=1, col=1)
fig.add_hline(y=0, line_width=1, line_color="black", row=2, col=1)

fig.update_layout(
    title=dict(
        text=(f"Composite Index Scores for {len(valid_words)}-Word Sample"),
        x=0.5,
    ),
    height=700,
    width=1000,
    margin=dict(l=60, r=40, t=80, b=60),
)

py.iplot(fig)

In [ ]:
# composite index ranking

composite_scores = feature_matrix_std @ pc1
composite_index = {valid_words[i]: float(composite_scores[i]) for i in range(len(valid_words))}
composite_index_sorted = sorted(composite_index.items(), key=lambda x: x[1], reverse=True)

N = 20 # num words to display for top/bottom composite index ranking

# extract top N most dynamic words
top_dynamic_entries = composite_index_sorted[:N]
top_words, top_scores = zip(*top_dynamic_entries)

# extract bottom N least dynamic words
bottom_dynamic_entries = composite_index_sorted[-N:]
bottom_words, bottom_scores = zip(*bottom_dynamic_entries)

# find max score across either top/bottom tails for symmetric horizontal scaling
interval_step = 0.5
top_scores_arr = np.asarray(top_scores, dtype=float)
bottom_scores_arr = np.asarray(bottom_scores, dtype=float)

x_pad_top = max(interval_step, 0.08 * float(np.max(top_scores_arr)))
x_pad_bot = max(interval_step, 0.08 * abs(float(np.min(bottom_scores_arr))))

x_max_top = float(np.max(top_scores_arr)) + x_pad_top
x_min_bot = float(np.min(bottom_scores_arr)) - x_pad_bot

# plot final rankings
fig, ax = plt.subplots(2, 1, figsize=(10, 10))
fig.suptitle(f"Composite Index Scores for Top & Bottom {N}-Word Subset", y=0.98)

# plotting top N most dynamic words
ax[0].barh(top_words, top_scores, color='salmon')
ax[0].invert_yaxis()
ax[0].set_title(f"Most Dynamic (Top {N})")
ax[0].set_xlabel('Composite Score (PC1)')
ax[0].set_xlim(0, x_max_top)
ax[0].axvline(0, color='black', linewidth=1)

# plotting bottom N least dynamic words
ax[1].barh(bottom_words, bottom_scores, color='cadetblue')
ax[1].invert_yaxis()
ax[1].set_title(f"Most Stable (Bottom {N})")
ax[1].set_xlabel('Composite Score (PC1)')
ax[1].set_xlim(x_min_bot, 0)
ax[1].axvline(0, color='black', linewidth=1)

fig.subplots_adjust(top=0.9, hspace=0.3)
plt.show()

In [ ]:
# print top/bottom N words by composite index score (long vertical list)

N = 50

print(f"Top {N} most dynamic words:")
for word, score in composite_index_sorted[:N]:
    print(f"{word}: {score:.4f}")

print(f"\nBottom {N} least dynamic words:")
for word, score in composite_index_sorted[-N:]:
    print(f"{word}: {score:.4f}")

In [ ]:
# print top/bottom N words by composite index score (wide horizontal block)

N = 100

print(f"Top {N} Most Dynamic Words:")
print(f"---------------------------")
for word, score in composite_index_sorted[:N]:
    print(f"{word}, ", end="")
print("...")

print(f"\nBottom {N} Least Dynamic Words:")
print(f"-------------------------------")
for word, score in composite_index_sorted[-N:]:
    print(f"{word}, ", end="")
print("...")

## Step 6: Visualization

In [ ]:
HOVER_TEMPLATE = "%{hovertext}<extra></extra>"

In [ ]:
def format_word_hover(word, decade, role=None):
    """Unified hover label for timeline + trajectory traces."""
    if role == "start":
        return f"{word} — START ({decade})"
    if role == "end":
        return f"{word} — END ({decade})"
    if role == "path":
        return f"{word} — interp path"
    return f"{word} ({decade})"

In [ ]:
def get_symmetric_cube_bounds(r3_data_matrices, step=1.0):
    """
    Finds absolute maximum across first 3 columns of all input matrices
    to create a uniform and symmetric cubic bound [-max, max] for all axes.

    Parameters:
    r3_data_matrices (list of numpy.ndarray or numpy.ndarray): Input data matrices.
    step (float): Step size for the tick marks.

    Returns:
    dict: Dictionary containing x/y/z ranges and dtick (same as step).
    """
    # if input is a single matrix, wrap it in a list
    if isinstance(r3_data_matrices, np.ndarray):
        r3_data_matrices = [r3_data_matrices]

    # look at first 3 columns (as that's the space we're plotting)
    global_max = 0
    for m in r3_data_matrices:
        # m[..., :3] handles both [words, dims] (single matrix) and [frames, words, dims] (array of matrices over time)
        current_abs_max = np.abs(m[..., :3]).max()
        global_max = max(global_max, current_abs_max)

    # snap global maximum to nearest tick step
    snapped_max = np.ceil(global_max / step) * step

    # add one step of padding if a data point is right on the edge
    if abs(snapped_max - global_max) < (step * 0.1):
        snapped_max += step

    bound = [-snapped_max, snapped_max]
    return {'x': bound, 'y': bound, 'z': bound, 'dtick': step}

#### Timeline plot:

In [ ]:
def plot_timeline_points(
    point_clouds,
    vocab_dict,
    decades,
    title,
    axis_ranges=None,
    camera=None,
    top_word_list=None,
    bottom_word_list=None,
    num_labeled_words_per_tail=0,
):
    marker_size=2
    default_point_color = 'lightgray';
    top_point_color = 'cadetblue';
    bottom_point_color = 'salmon';

    n_timeline_frames = point_clouds.shape[0]
    nd_decades = len(decades)
    frames = []

    # map each vocab row (fixed across decades) to a marker color
    base_pos = {w: i for i, w in enumerate(vocab_dict[decades[0]])}
    n_pts = len(base_pos)

    top_words = list(top_word_list or [])
    bottom_words = list(bottom_word_list or [])
    top_set = set(top_words)
    bottom_set = set(bottom_words)

    # default: no top/bottom words --> entire cloud stays blue
    # alt: N top/bottom words --> top is teal blue, middle is lightgray, bottom is salmon red
    use_tail_colors = bool(top_words or bottom_words)

    if use_tail_colors:
        # N top/bottom words => teal top, gray middle, salmon bottom
        point_colors = [default_point_color] * n_pts
        for w in top_words:
            if w in base_pos:
                point_colors[base_pos[w]] = top_point_color
        for w in bottom_words:
            if w in base_pos:
                point_colors[base_pos[w]] = bottom_point_color

        k = max(0, int(num_labeled_words_per_tail))
        labeled_words = top_words[:k] + bottom_words[:k]
    else:
        point_colors = 'blue'
        labeled_words = []

    label_indices = [base_pos[w] for w in labeled_words if w in base_pos]

    # default: no word label tags and axes spikes rendered
    # alt: k word label tags and axes spikes (only up to top/bottom N words or below) rendered
    k = max(0, int(num_labeled_words_per_tail))
    labeled_words = top_words[:k] + bottom_words[-k:]
    label_indices = [base_pos[w] for w in labeled_words if w in base_pos]

    # word label tag box color matches top/bottom color
    def tag_color_for(w):
        if w in top_set:
            return top_point_color
        if w in bottom_set:
            return bottom_point_color
        return default_point_color

    base_annotations = []  # frame-0 pins for the initial (pre-play) layout

    # create frames for each interpolated point cloud
    for frame_idx, frame_data in enumerate(point_clouds):
        # map frame index to nearest decade label (uniform t grid from interpolate_sequence)
        if n_timeline_frames <= 1:
            decade_idx = 0
        else:
            decade_idx = int(round(frame_idx * (nd_decades - 1) / (n_timeline_frames - 1)))

        # clamp decade index to valid range
        decade_idx = max(0, min(decade_idx, nd_decades - 1))
        curr_decade = decades[decade_idx]

        words = vocab_dict[curr_decade]
        frame_traces = [
            go.Scatter3d(
                x=frame_data[:, 0],
                y=frame_data[:, 1],
                z=frame_data[:, 2],
                mode='markers',
                marker=dict(size=marker_size, color=point_colors),
                hovertext=[format_word_hover(w, curr_decade) for w in words],
                hovertemplate=HOVER_TEMPLATE,
                name=f"Frame {frame_idx}",
            )
        ]

        # build pin tags for the tracked words
        frame_annotations = []
        if label_indices:
            lf = frame_data[label_indices]

            if axis_ranges:
                B = axis_ranges["x"][1]  # symmetric bound magnitude
                sx, sy, sz = [], [], []
                def seg(p0, p1):
                    sx.extend([p0[0], p1[0], None])
                    sy.extend([p0[1], p1[1], None])
                    sz.extend([p0[2], p1[2], None])
                for (lx, ly, lz) in lf:
                    # perpendicular droplines from point to each back wall
                    seg((lx, ly, lz), (-B, ly, lz))   # to x-wall
                    seg((lx, ly, lz), (lx, -B, lz))   # to y-wall
                    seg((lx, ly, lz), (lx, ly, -B))   # to z-wall
                    # projected crosshair on x-wall (x = -B)
                    seg((-B, ly, lz), (-B, -B, lz))
                    seg((-B, ly, lz), (-B, ly, -B))
                    # projected crosshair on y-wall (y = -B)
                    seg((lx, -B, lz), (-B, -B, lz))
                    seg((lx, -B, lz), (lx, -B, -B))
                    # projected crosshair on z-wall (z = -B)
                    seg((lx, ly, -B), (-B, ly, -B))
                    seg((lx, ly, -B), (lx, -B, -B))

                frame_traces.append(
                    go.Scatter3d(
                        x=sx, y=sy, z=sz,
                        mode='lines',
                        line=dict(color='rgba(0,0,0,0.25)', width=2),
                        hoverinfo='skip',
                        showlegend=False,
                        name='spikes',
                    )
                )

            # boxed pin tag per tracked word
            for (lx, ly, lz), w in zip(lf, labeled_words):
                tc = tag_color_for(w)
                frame_annotations.append(dict(
                    x=lx, y=ly, z=lz,
                    text=w,
                    showarrow=True,
                    arrowhead=2,
                    arrowsize=1,
                    arrowwidth=1.5,
                    arrowcolor=tc,
                    ax=0, ay=-40, # pixel offset of the word box tag from the datapoint
                    font=dict(color='white', size=11),
                    bgcolor=tc,
                    bordercolor='white',
                    borderwidth=1,
                    borderpad=3,
                ))

        if frame_idx == 0:
            base_annotations = frame_annotations

        frame = go.Frame(
            data=frame_traces,
            name=f"Frame {frame_idx}",
            layout=go.Layout(scene=dict(annotations=frame_annotations)),
        )
        frames.append(frame)

    # build slider steps (unfortunately non interpolated)
    _slider_steps = []
    for i, decade in enumerate(decades):
        if n_timeline_frames <= 1:
            k = 0
        else:
            k = int(round(i * (n_timeline_frames - 1) / (nd_decades - 1)))
        _slider_steps.append({
            "args": [[f"Frame {k}"], {
                "frame": {"duration": 100, "redraw": True},
                "mode": "immediate",
                "transition": {"duration": 100},
            }],
            "label": decade,
            "method": "animate",
        })

    slider_config = [{
        "steps": _slider_steps,
        "active": 0,
        "currentvalue": dict(visible=False),
        "x": 0.1,
        "xanchor": "left",
        "y": 0,
        "yanchor": "top",
        "len": 0.8,
    }]

    # base scene configuration (if global bounds were provided, apply them)
    scene_config = dict(
        xaxis=dict(title="PC1", range=axis_ranges["x"], dtick=axis_ranges["dtick"], autorange=False, showspikes=True),
        yaxis=dict(title="PC2", range=axis_ranges["y"], dtick=axis_ranges["dtick"], autorange=False, showspikes=True),
        zaxis=dict(title="PC3", range=axis_ranges["z"], dtick=axis_ranges["dtick"], autorange=False, showspikes=True),
        aspectmode="cube",
        annotations=base_annotations,
        **({"camera": camera} if camera is not None else {}),
    ) if axis_ranges else dict(
        xaxis=dict(title="PC1", autorange=True, showspikes=True),
        yaxis=dict(title="PC2", autorange=True, showspikes=True),
        zaxis=dict(title="PC3", autorange=True, showspikes=True),
        aspectmode="cube",
        annotations=base_annotations,
        **({"camera": camera} if camera is not None else {}),
    )

    menu_config = [{
        "buttons": [
            {"args": [None, {"frame": {"duration": 100, "redraw": True}, "mode": "immediate", "fromcurrent": True}], "label": "Play", "method": "animate"},
            {"args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}], "label": "Pause", "method": "animate"},
            {"args": [[f"Frame 0"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}], "label": "Reset", "method": "animate"},
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "type": "buttons",
        "x": 0.5,
        "xanchor": "center",
        "y": 0,
        "yanchor": "top",
    }]

    # define layout for timeline plot
    layout = go.Layout(
        title=title,
        title_x=0.5,
        margin=dict(l=0, r=0, b=0, t=40),
        scene=scene_config,
        height=600, width=800,
        sliders=slider_config,
        updatemenus=menu_config,
        showlegend=False,
    )

    # plot word data points
    fig = go.Figure(data=list(frames[0].data), layout=layout, frames=frames)
    py.iplot(fig)

In [ ]:
# initial plot setup and parameters

N = 100 # top/bottom dynamic words to color in the plot
num_labeled_words_per_tail = 2  # smaller set of top/bottom dynamic words to label in the plot
num_interps = 15  # adjust for smoother transitions here
step_size = 5.0

# camera zoom level control (default is x=1, y=1, z=1)
camera = dict(eye=dict(x=1.5, y=1.5, z=1.5))

# extract top/bottom word lists from our sorted composite index
top_n_words = [w for w, s in composite_index_sorted[:N]] if N > 0 else []
bottom_n_words = [w for w, s in composite_index_sorted[-N:]] if N > 0 else []

# stack all projected R3 embeddings into single array
# then interpolate R3 embedding point clouds along time axis
proj_embeddings_array = [proj_embeddings_dict[d] for d in decades]
point_clouds = interpolate_sequence(proj_embeddings_array, num_interpolations=num_interps)

# calculate global bounds for consistent and symmetric scaling
timeline_plot_bounds = get_symmetric_cube_bounds([point_clouds], step=step_size)

In [ ]:
# plot timeline points
plot_timeline_points(
    point_clouds,
    vocab_dict,
    decades,
    f"Word Position Timeline for {sample_size}-Word Sample",
    axis_ranges=timeline_plot_bounds,
    camera=camera,
)

In [ ]:
# plot timeline points with top/bottom tails color coded
plot_timeline_points(
    point_clouds,
    vocab_dict,
    decades,
    f"Word Position Timeline for {sample_size}-Word Sample (Color Coded)",
    axis_ranges=timeline_plot_bounds,
    camera=camera,
    top_word_list=top_n_words,
    bottom_word_list=bottom_n_words,
    num_labeled_words_per_tail=num_labeled_words_per_tail,
)

#### Trajectory plots:

In [ ]:
def plot_trajectory_group(
    trajectories,
    word_list,
    decades,
    title,
    axis_ranges=None,
    camera=None,
):
    marker_size=2
    trace_color_palette = plc.qualitative.Prism
    traces = []

    for i, word in enumerate(word_list):
        # get raw decade points (e.g., 11 points for 1880-1980)
        points = trajectories[word]

        # parameterize t from 0 to 1
        t_steps = np.linspace(0, 1, len(decades))
        t_fine = np.linspace(0, 1, 100) # 100 points for smooth interpolation

        # interpolate each dimension (x, y, z)
        cs_x = CubicSpline(t_steps, points[:, 0])
        cs_y = CubicSpline(t_steps, points[:, 1])
        cs_z = CubicSpline(t_steps, points[:, 2])

        x_fine = cs_x(t_fine)
        y_fine = cs_y(t_fine)
        z_fine = cs_z(t_fine)

        # color for this specific word
        color = trace_color_palette[i % len(trace_color_palette)]
        points = trajectories[word]

        # smooth trajectory path
        traces.append(go.Scatter3d(
            x=x_fine, y=y_fine, z=z_fine,
            mode='lines',
            line=dict(width=3, color=color),
            name=word,
            legendgroup=word,
            hoverinfo='skip',
        ))

        # decade markers (keyframes)
        traces.append(go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode='markers',
            marker=dict(size=marker_size, color=color, symbol='circle', opacity=0.8),
            name=word,
            legendgroup=word,
            showlegend=False,
            hovertext=[format_word_hover(word, d) for d in decades],
            hovertemplate=HOVER_TEMPLATE,
        ))

        # start/end indicators
        traces.append(go.Scatter3d(
            x=[points[0, 0], points[-1, 0]],
            y=[points[0, 1], points[-1, 1]],
            z=[points[0, 2], points[-1, 2]],
            mode='markers',
            hovertext=[
                format_word_hover(word, decades[0], role="start"),
                format_word_hover(word, decades[-1], role="end"),
            ],
            hovertemplate=HOVER_TEMPLATE,
            marker=dict(size=marker_size, color='black', symbol='diamond'),
            showlegend=False,
            legendgroup=word,
        ))

    # base scene configuration (if global bounds were provided, apply them)
    scene_config = dict(
        xaxis=dict(title="PC1", range=axis_ranges["x"], dtick=axis_ranges["dtick"], autorange=False),
        yaxis=dict(title="PC2", range=axis_ranges["y"], dtick=axis_ranges["dtick"], autorange=False),
        zaxis=dict(title="PC3", range=axis_ranges["z"], dtick=axis_ranges["dtick"], autorange=False),
        aspectmode="cube",
        **({"camera": camera} if camera is not None else {}),
    ) if axis_ranges else dict(
        xaxis=dict(title="PC1", autorange=True),
        yaxis=dict(title="PC2", autorange=True),
        zaxis=dict(title="PC3", autorange=True),
        aspectmode="cube",
        **({"camera": camera} if camera is not None else {}),
    )

    layout = go.Layout(
        title=title,
        title_x=0.5,
        scene=scene_config,
        height=600, width=800,
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig = go.Figure(data=traces, layout=layout)
    py.iplot(fig)

In [ ]:
# initial plot setup and parameters

N = 15 # number of words to plot
step_size = 5.0

# camera zoom level control (default is x=1, y=1, z=1)
camera = dict(eye=dict(x=1.6, y=1.6, z=1.6))

# extract word lists from our sorted composite index
top_n_words = [w for w, s in composite_index_sorted[:N]]
bottom_n_words = [w for w, s in composite_index_sorted[-N:]]

# stack all R3 trajectories for the words we're about to plot
word_list = top_n_words + bottom_n_words # size 2N
trajectories = np.vstack([proj_word_trajectories[w] for w in word_list])

# calculate global bounds for consistent and symmetric scaling
trajectory_plot_bounds = get_symmetric_cube_bounds([trajectories], step=step_size)

In [ ]:
# plot top movers
plot_trajectory_group(
    proj_word_trajectories,
    top_n_words,
    decades,
    f"Word Trajectories for Top {N} Most Dynamic Words",
    axis_ranges=trajectory_plot_bounds,
    camera=camera,
)

In [ ]:
# plot most stable
plot_trajectory_group(
    proj_word_trajectories,
    bottom_n_words,
    decades,
    f"Word Trajectories for Bottom {N} Most Stable Words",
    axis_ranges=trajectory_plot_bounds,
    camera=camera,
)